# Day 28: How LLMs Actually Work: RNNs and Transformers

You have built two bots:

- **Day 25:** the old way. Bag of words. It **forgets the order** of the words.
- **Day 27:** the LLM way. It understands long chats.

Today we look **behind the API**: how does a model read a sentence **in order**? We go back to the Day 23 sentence:

> **"i drink sweet ___"** → **tea**

We read it in 2 ways: the **RNN** way (old), and the **Transformer** way (inside every LLM today).

## 📋 Agenda

```
TOP     Step 6   Weekly checkpoint        build + explain
          ▲
        Step 5   Behind your API call     the full path of one message
          ▲
        Step 4   Transformer              all words look at all words, at once
          ▲
        Step 3   RNN problems             forgets far words, and it is slow
          ▲
        Step 2   RNN                      read one word at a time, keep a memory
          ▲
BOTTOM  Step 1   Order matters            bag of words is not enough
```

### New words today

| Word | Meaning |
|---|---|
| **Sequence** | Things in order, one after another. A sentence is a sequence of words. |
| **RNN** | Recurrent Neural Network. It reads **one word at a time** and keeps a memory. |
| **Hidden state** | The RNN's memory: a vector that is updated after every word. |
| **Self-attention** | Every word looks at every other word, and takes what is useful (Day 23 Layer 4). |
| **Parallel** | Many jobs done **at the same time**, not one after another. |
| **Transformer** | The network design inside LLMs. Its heart is self-attention. |

---
# Class Problem: read "i drink sweet" in order

## Step 1: Order matters

Bag of words (Day 25) only **counts**. These 2 sentences get the **same** numbers:

| Sentence | Meaning |
|---|---|
| *Ali pays Sara* | Ali gives the money. |
| *Sara pays Ali* | Sara gives the money. |

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

bow = CountVectorizer()
print(bow.fit_transform(["Ali pays Sara", "Sara pays Ali"]).toarray())   # the same 2 rows!

A model for language must know the **order**. Here are 2 ways to do that.

We use the same tools as Day 23: the vocabulary, the embedding table `E` and `softmax`.

In [ ]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)

vocab = ["i", "drink", "eat", "sweet", "tea", "apple"]
#             person drink food sweet
E = np.array([[3,     0,    0,   0],    # 0 i
              [0,     2,    0,   0],    # 1 drink
              [0,     0,    2,   0],    # 2 eat
              [0,     1,    1,   1],    # 3 sweet
              [0,     2,    0,   1],    # 4 tea
              [0,     0,    2,   1]],   # 5 apple
             dtype=float)

def softmax(scores):
    e = np.exp(scores - scores.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)      # each row adds up to 1

## Step 2: RNN: read one word at a time

- The RNN reads the words **in order**: first `i`, then `drink`, then `sweet`.
- It keeps a **memory** vector, the **hidden state** `h`.
- After each word: **new memory = part of the old memory + the new word**.

*Everyday example:* you listen to a long story. You cannot remember every word, so you keep a **short note in your head**. After each sentence you change the note a little.

Our RNN keeps **half** of the old memory (`keep = 0.5`). A real RNN **learns** how much to keep, but the idea is the same.

In [ ]:
def rnn(words, keep=0.5):
    h = np.zeros(4)                                   # empty memory at the start
    for w in words:
        h = keep * h + E[vocab.index(w)]              # old memory (a part) + the new word
        print(f"after {w:6} h = {h}")
    return h

h = rnn(["i", "drink", "sweet"])

At the end, `h` has a bit of **every** word, and the **order** changes the numbers. The model uses `h` to guess the next word: the drink number is big, so the guess is **tea**.

In [ ]:
for word in ["tea", "apple"]:
    print(word, "score:", np.dot(h, E[vocab.index(word)]))      # tea wins

**✏️ Quick Check:** Run `rnn` on `["i", "eat", "sweet"]`. Which word wins now: tea or apple?

In [ ]:
# your code here

**Answer:** apple. Now the **food** number in `h` is big, not the drink number.

In [ ]:
h2 = rnn(["i", "eat", "sweet"])
for word in ["tea", "apple"]:
    print(word, "score:", np.dot(h2, E[vocab.index(word)]))

## Step 3: RNN problems

**Problem 1: it forgets far words.** After every word, the old memory is cut in half. A word far back becomes very small.

Look at how much of one word is **left** after more words come:

In [ ]:
for n in [1, 2, 5, 10, 20]:
    print(f"after {n:2} more words: {0.5 ** n:.6f} of the word is left")

*"**I drink** my chai every morning at the small shop near my home with my friends, and I like it very ___"*

The clue **drink** is far from the blank. In the RNN memory it is almost **gone**.

**Problem 2: it is slow.** Word 2 must wait for word 1. Word 3 must wait for word 2. It cannot read all the words **at the same time**.

*Everyday example:* a shop with **one cashier**. 100 customers wait in **one line**.

**✏️ Quick Check:** A better RNN keeps 0.9 of the memory. How much of a word is left after 10 more words? Use code.

In [ ]:
# your code here

**Answer:** about 0.35. Better than 0.5 (0.001 left), but the far word is still **weaker** than the near words.

In [ ]:
print(0.9 ** 10)

## Step 4: Transformer: all words look at all words, at once

The Transformer does **not** keep a memory note. **Every word looks at every word directly**, with **self-attention** (Day 23 Layer 4):

1. **Scores:** dot product of every word with every word. `X @ X.T` does all of them in **one** step.
2. **Weights:** softmax on each row. Each row adds up to 1.
3. **New vectors:** each word takes a **mix** of the words it looked at.

*Everyday example:* not one cashier, but **every customer at their own counter, at the same time**. And every word can "see" the far words as clearly as the near ones.

In [ ]:
words = ["i", "drink", "sweet"]
X = E[[vocab.index(w) for w in words]]      # 3 rows, one per word

scores  = X @ X.T                           # 1. every word · every word, all at once
weights = softmax(scores)                   # 2. each row → chances that add up to 1
new_X   = weights @ X                       # 3. each word takes a mix of the others

print("weights (row = the word that looks):")
for w, row in zip(words, weights):
    print(f"{w:6}", row)

Read the **sweet** row: it looks at itself and at **drink**, and almost not at **i**. So the new vector of `sweet` now has the **drink** idea in it, and the guess is **tea**.

**No loop over the words.** One matrix multiply did all the words together. This is **parallel**, and it is why GPUs can train huge Transformers.

In [ ]:
for word in ["tea", "apple"]:
    print(word, "score:", np.dot(new_X[-1], E[vocab.index(word)]))    # use the new "sweet" vector

**✏️ Quick Check:** Print the **drink** row of `weights`. Which word does `drink` look at most?

In [ ]:
# your code here

**Answer:** itself (drink). Its own dot product is the biggest.

In [ ]:
print(weights[1])

### RNN vs Transformer

| | RNN | Transformer |
|---|---|---|
| How it reads | One word at a time | All words at the same time |
| Far words | Fade away | Seen as clearly as near words |
| Speed | Slow: each word waits | Fast: parallel on GPUs |
| Used in | Older translators, keyboards | ChatGPT, Claude, Gemini, our free model |

## Step 5: Behind your API call

When your Day 27 bot sends *"How much is a samosa?"*, this happens on the other computer:

```
"How much is a samosa?"
   │  1. Tokenize            text → tokens → IDs                    (Day 23, Day 25)
   ▼
[2437, 1790, 374, ...]
   │  2. Embeddings          each ID → a vector                     (Day 23, Day 24)
   ▼
   │  3. Self-attention      every token looks at every token       (today)
   │     × many layers       (big models: around 100 layers)
   ▼
   │  4. Softmax             scores → chances for the next token    (Day 23)
   ▼
"A"  →  add it, and repeat for the next token, until the answer is done
```

**This is why:**

- we pay for **tokens** (step 1);
- there is a **context limit**: attention compares every token with every token, so long text costs a lot;
- **temperature 0** gives the same answer: it always picks the top token in step 4.

**✏️ Quick Check:** Why does a 10-page document cost more than a 1-page document, even if the answer is short?

In [ ]:
# your code here

**Answer:** We pay for **input tokens** too. 10 pages = many more tokens, and attention must compare all of them with each other.

### Key takeaways

- Language needs **order**. Bag of words loses it.
- An **RNN** reads one word at a time and keeps a **hidden state** (memory). Far words **fade**, and it is **slow**.
- A **Transformer** uses **self-attention**: every word looks at every word, **at the same time**.
- This is what is inside the API: tokens → embeddings → attention (many layers) → softmax → next token → repeat.

---
# ✅ Weekly Skill Checkpoint

**"I can build a working LLM-powered application and explain how it works under the hood."**

### Part 1: Build (show your teacher)

- [ ] Your Streamlit app from Day 27 runs with `streamlit run ...`
- [ ] It has a good system prompt (5 parts: role, facts, rules, style, unknown).
- [ ] It remembers: ask a question, then a follow-up that needs the first answer.
- [ ] It says "I don't know" for a question outside its facts.
- [ ] The key is in `.env`, not in the code.

### Part 2: Explain (write 1–2 lines for each)

1. Why do we send the **whole history** in every request?
2. What is a **token**, and why do we pay for tokens?
3. What does **self-attention** do?
4. Why is a Transformer **faster** than an RNN?
5. Your Day 25 bot and your Day 27 bot both get *"dilivery charges?"*. Which one answers well, and why?
6. What is a **hallucination**? How does your system prompt try to stop it?

In [ ]:
# Part 2: write your answers here (or in a markdown cell)